In [ ]:
#https://chatgpt.com/g/g-p-676c80353b988191819d6d02aca806d6/c/69554944-cea4-8331-9f60-b3e1a095fbed

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ============================================================
# PATHS
# ============================================================

VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'
AREAS_PATH  = '/home/maria/ProjectionSort/data/brain_area.npy'

# ============================================================
# LOAD DATA
# ============================================================

vit   = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
R     = np.load(NEURAL_PATH).T                                   # (images, neurons)
areas = np.load(AREAS_PATH, allow_pickle=True)

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

# ============================================================
# ANIMATE / INANIMATE LABEL
# ============================================================

# ImageNet top-1 class indices (assumed available)
# classes 0–397 = animate, 398–999 = inanimate
top1 = np.argmax(vit, axis=1)
y = (top1 <= 397).astype(int)   # shape (images,)

print("Animate fraction:", y.mean())

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import itertools

# ============================================================
# PATHS
# ============================================================

VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'
AREAS_PATH  = '/home/maria/ProjectionSort/data/brain_area.npy'

# ============================================================
# LOAD DATA
# ============================================================

vit   = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
R     = np.load(NEURAL_PATH).T                                   # (images, neurons)
areas = np.load(AREAS_PATH, allow_pickle=True)

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

# ============================================================
# ANIMATE / INANIMATE LABEL
# ============================================================

top1 = np.argmax(vit, axis=1)
y = (top1 <= 397).astype(int)
print("Animate fraction:", y.mean())

# ============================================================
# STANDARDIZE
# ============================================================

R = StandardScaler().fit_transform(R)

# ============================================================
# FISHER TRACE FUNCTION (NO FULL MATRIX)
# ============================================================

def fisher_trace(X, y, C=1.0):
    clf = LogisticRegression(
        penalty="l2",
        C=C,
        fit_intercept=True,
        solver="lbfgs",
        max_iter=500
    )
    clf.fit(X, y)

    p = clf.predict_proba(X)[:, 1]
    w = p * (1 - p)

    # Trace(F) = sum_i p_i (1-p_i) ||x_i||^2
    trace = np.sum(w[:, None] * (X ** 2))
    return trace

# ============================================================
# GLOBAL FISHER TRACE
# ============================================================

print("\nComputing global Fisher trace...")
trace_global = fisher_trace(R, y)
print("Trace(global):", trace_global)

# ============================================================
# REGION-WISE FISHER TRACE
# ============================================================

region_traces = {}
unique_regions = np.unique(areas)

print("\nComputing region-wise Fisher traces...")
for r in unique_regions:
    idx = np.where(areas == r)[0]
    Xr = R[:, idx]
    tr = fisher_trace(Xr, y)
    region_traces[r] = tr
    print(f"Region {r:>6s} | neurons: {len(idx):>5d} | trace: {tr:.3e}")

# ============================================================
# TRACE RATIOS
# ============================================================

print("\nTrace ratios (region / global):")
trace_ratios = {r: tr / trace_global for r, tr in region_traces.items()}

for r, val in sorted(trace_ratios.items(), key=lambda x: -x[1]):
    print(f"Region {r:>6s} | ratio: {val:.4f}")

# ============================================================
# PAIRWISE COHERENCE RATIOS
# ============================================================

print("\nComputing pairwise region coherence ratios...")

pairwise_ratios = {}
for r1, r2 in itertools.combinations(unique_regions, 2):
    idx = np.where((areas == r1) | (areas == r2))[0]
    Xpair = R[:, idx]

    tr_pair = fisher_trace(Xpair, y)
    denom = region_traces[r1] + region_traces[r2]
    ratio = tr_pair / denom if denom > 0 else np.nan

    pairwise_ratios[(r1, r2)] = ratio
    print(f"{r1:>6s} + {r2:>6s} | coherence ratio: {ratio:.4f}")

# ============================================================
# SAVE RESULTS
# ============================================================

np.save("fisher_region_traces.npy", region_traces)
np.save("fisher_trace_ratios.npy", trace_ratios)
np.save("fisher_pairwise_coherence.npy", pairwise_ratios)

print("\nSaved:")
print(" - fisher_region_traces.npy")
print(" - fisher_trace_ratios.npy")
print(" - fisher_pairwise_coherence.npy")


Images: 118
Neurons: 39209
Animate fraction: 0.5338983050847458

Computing global Fisher trace...
Trace(global): 807.4684314521197

Computing region-wise Fisher traces...
Region  VISal | neurons:  4249 | trace: 7.106e+02
Region  VISam | neurons:  2040 | trace: 6.629e+02
Region   VISl | neurons:  8323 | trace: 7.389e+02
Region   VISp | neurons: 14382 | trace: 8.655e+02
Region  VISpm | neurons:  4771 | trace: 7.283e+02
Region  VISrl | neurons:  5444 | trace: 7.658e+02

Trace ratios (region / global):
Region   VISp | ratio: 1.0719
Region  VISrl | ratio: 0.9484
Region   VISl | ratio: 0.9151
Region  VISpm | ratio: 0.9019
Region  VISal | ratio: 0.8800
Region  VISam | ratio: 0.8210

Computing pairwise region coherence ratios...
 VISal +  VISam | coherence ratio: 0.5230
 VISal +   VISl | coherence ratio: 0.5403
 VISal +   VISp | coherence ratio: 0.5441
 VISal +  VISpm | coherence ratio: 0.5126
 VISal +  VISrl | coherence ratio: 0.5366
 VISam +   VISl | coherence ratio: 0.5725
 VISam +   VISp |

In [2]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import itertools

# ============================================================
# PATHS
# ============================================================

VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'
AREAS_PATH  = '/home/maria/ProjectionSort/data/brain_area.npy'

# ============================================================
# LOAD DATA
# ============================================================

vit   = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']   # (images, 1000)
R     = np.load(NEURAL_PATH).T                                   # (images, neurons)
areas = np.load(AREAS_PATH, allow_pickle=True)

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

# ============================================================
# ANIMATE / INANIMATE LABEL
# ============================================================

top1 = np.argmax(vit, axis=1)
y = (top1 <= 397).astype(int)
print("Animate fraction:", y.mean())

# ============================================================
# STANDARDIZE ONCE GLOBALLY
# ============================================================

R = StandardScaler().fit_transform(R)

# ============================================================
# FIT GLOBAL MODEL (FREEZE GEOMETRY)
# ============================================================

clf_global = LogisticRegression(
    penalty="l2",
    C=1.0,
    fit_intercept=True,
    solver="lbfgs",
    max_iter=500
)
clf_global.fit(R, y)

p_global = clf_global.predict_proba(R)[:, 1]
w_global = p_global * (1 - p_global)     # frozen Fisher weights

# ============================================================
# AVERAGE FISHER TRACE FUNCTION (DIMENSION-FREE)
# ============================================================

def avg_fisher_trace(X, w):
    """
    Average Fisher trace per parameter direction:
    (1/d) * sum_i w_i ||x_i||^2
    """
    d = X.shape[1]
    return np.sum(w[:, None] * (X ** 2)) / d

# ============================================================
# GLOBAL AVERAGE TRACE
# ============================================================

avg_trace_global = avg_fisher_trace(R, w_global)
print("\nAverage Fisher trace (global):", avg_trace_global)

# ============================================================
# REGION-WISE (GLOBAL-FROZEN) AVERAGE TRACES
# ============================================================

region_avg_traces = {}
unique_regions = np.unique(areas)

print("\nRegion-wise average Fisher traces (global-frozen):")
for r in unique_regions:
    idx = np.where(areas == r)[0]
    Xr = R[:, idx]

    avg_tr = avg_fisher_trace(Xr, w_global)
    region_avg_traces[r] = avg_tr

    print(
        f"Region {r:>6s} | neurons: {len(idx):>5d} | "
        f"avg trace: {avg_tr:.4f}"
    )

# ============================================================
# REGION / GLOBAL RATIOS (MEAN CURVATURE CONTRIBUTION)
# ============================================================

print("\nRegion / global average-trace ratios:")
region_ratios = {
    r: tr / avg_trace_global for r, tr in region_avg_traces.items()
}

for r, val in sorted(region_ratios.items(), key=lambda x: -x[1]):
    print(f"Region {r:>6s} | ratio: {val:.4f}")

# ============================================================
# PAIRWISE COHERENCE (PAIRWISE-FROZEN GEOMETRY)
# ============================================================

print("\nPairwise coherence (average-trace based):")

pairwise_coherence = {}

for r1, r2 in itertools.combinations(unique_regions, 2):
    idx_pair = np.where((areas == r1) | (areas == r2))[0]
    Xpair = R[:, idx_pair]

    # fit pairwise model → freeze its geometry
    clf_pair = LogisticRegression(
        penalty="l2",
        C=1.0,
        fit_intercept=True,
        solver="lbfgs",
        max_iter=500
    )
    clf_pair.fit(Xpair, y)

    p_pair = clf_pair.predict_proba(Xpair)[:, 1]
    w_pair = p_pair * (1 - p_pair)

    # average traces under same geometry
    avg_pair = avg_fisher_trace(Xpair, w_pair)

    X1 = R[:, areas == r1]
    X2 = R[:, areas == r2]

    avg_r1 = avg_fisher_trace(X1, w_pair)
    avg_r2 = avg_fisher_trace(X2, w_pair)

    coherence = avg_pair / (0.5 * (avg_r1 + avg_r2))
    pairwise_coherence[(r1, r2)] = coherence

    print(f"{r1:>6s} + {r2:>6s} | coherence: {coherence:.4f}")

# ============================================================
# SAVE RESULTS
# ============================================================

np.save("fisher_avgtrace_global.npy", avg_trace_global)
np.save("fisher_avgtrace_regions.npy", region_avg_traces)
np.save("fisher_avgtrace_region_ratios.npy", region_ratios)
np.save("fisher_avgtrace_pairwise_coherence.npy", pairwise_coherence)

print("\nSaved:")
print(" - fisher_avgtrace_global.npy")
print(" - fisher_avgtrace_regions.npy")
print(" - fisher_avgtrace_region_ratios.npy")
print(" - fisher_avgtrace_pairwise_coherence.npy")


Images: 118
Neurons: 39209
Animate fraction: 0.5338983050847458

Average Fisher trace (global): 0.020593956271573358

Region-wise average Fisher traces (global-frozen):
Region  VISal | neurons:  4249 | avg trace: 0.0198
Region  VISam | neurons:  2040 | avg trace: 0.0198
Region   VISl | neurons:  8323 | avg trace: 0.0205
Region   VISp | neurons: 14382 | avg trace: 0.0207
Region  VISpm | neurons:  4771 | avg trace: 0.0203
Region  VISrl | neurons:  5444 | avg trace: 0.0218

Region / global average-trace ratios:
Region  VISrl | ratio: 1.0584
Region   VISp | ratio: 1.0031
Region   VISl | ratio: 0.9952
Region  VISpm | ratio: 0.9842
Region  VISam | ratio: 0.9627
Region  VISal | ratio: 0.9597

Pairwise coherence (average-trace based):
 VISal +  VISam | coherence: 0.9962
 VISal +   VISl | coherence: 1.0021
 VISal +   VISp | coherence: 0.9965
 VISal +  VISpm | coherence: 1.0006
 VISal +  VISrl | coherence: 1.0036
 VISam +   VISl | coherence: 0.9993
 VISam +   VISp | coherence: 0.9924
 VISam +  V

In [3]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# ============================================================
# PATHS
# ============================================================

VIT_PATH    = '/home/maria/ProjectionSort/data/google_vit-base-patch16-224_embeddings_logits.pkl'
NEURAL_PATH = '/home/maria/ProjectionSort/data/hybrid_neural_responses_reduced.npy'

# ============================================================
# LOAD DATA
# ============================================================

vit = np.load(VIT_PATH, allow_pickle=True)['natural_scenes']
R   = np.load(NEURAL_PATH).T

print("Images:", vit.shape[0])
print("Neurons:", R.shape[1])

# ============================================================
# LABEL
# ============================================================

top1 = np.argmax(vit, axis=1)
y = (top1 <= 397).astype(int)

# ============================================================
# STANDARDIZE
# ============================================================

R = StandardScaler().fit_transform(R)

# ============================================================
# FIT LOGISTIC REGRESSION
# ============================================================

clf = LogisticRegression(
    penalty="l2",
    C=1.0,
    fit_intercept=True,
    solver="lbfgs",
    max_iter=500
)
clf.fit(R, y)

w = clf.coef_.ravel()
w = w / np.linalg.norm(w)   # unit axis

# ============================================================
# COMPUTE FISHER CURVATURE ALONG AXIS
# ============================================================

p = clf.predict_proba(R)[:, 1]
weights = p * (1 - p)

# lambda_axis = w^T F w = sum_i p_i(1-p_i) (x_i·w)^2
proj = R @ w
lambda_axis = np.sum(weights * proj**2)

# ============================================================
# BASELINE: RANDOM DIRECTIONS
# ============================================================

n_random = 20
rand_vals = []

for _ in range(n_random):
    v = np.random.randn(R.shape[1])
    v /= np.linalg.norm(v)
    rand_vals.append(np.sum(weights * (R @ v)**2))

rand_vals = np.array(rand_vals)

# ============================================================
# REPORT
# ============================================================

print("\n=== AXIS CURVATURE CHECK ===")
print("Curvature along learned axis:", lambda_axis)
print("Random direction mean:", rand_vals.mean())
print("Random direction std :", rand_vals.std())
print("Axis / random mean ratio:", lambda_axis / rand_vals.mean())


Images: 118
Neurons: 39209

=== AXIS CURVATURE CHECK ===
Curvature along learned axis: 6.55534190070358
Random direction mean: 0.0215847471006487
Random direction std : 0.0027086127357066705
Axis / random mean ratio: 303.7025113213658
